# AlphaKhulnasoft — Tensor Matrix Multiplication

Exact, API-free study of matrix-multiplication algorithms as bilinear
(rank-one) factorizations. Runs from a clean Colab runtime after
`!pip install alphakhulnasoft` (or by mounting the repo).

In [ ]:
import alphakhulnasoft.matmul as mm
from alphakhulnasoft.matmul.algorithms import loader, registry
from alphakhulnasoft.matmul.reference import verify_exact

print("Registered algorithms:", registry.names())

## 1. Load & inspect a factorization

Factorizations are versioned JSON (format 1.0) recording dimensions, field,
rank, factor tensors, and provenance. Load one from the checked-in data dir
or the in-memory registry.

In [ ]:
alg = loader.load_local("alphakhulnasoft/matmul/data/schoolbook_2x2.json")
print("algorithm :", alg.algorithm_name)
print("dims      :", alg.dims)
print("field     :", alg.field.value)
print("rank      :", alg.rank)
print("factors   :", len(alg.factors))
print("provenance:", alg.provenance)

## 2. Exact correctness

Reconstruction uses `Fraction` arithmetic for exact fields, so correctness is
**proven**, not approximated. The same check holds for randomized inputs.

In [ ]:
import random

a = [[1, 2], [3, 4]]
b = [[5, 6], [7, 8]]
print("strassen_2x2 exact?  ", verify_exact(mm.registry.get("strassen_2x2"), a, b))
print("schoolbook_2x2 exact?", verify_exact(mm.registry.get("schoolbook_2x2x2"), a, b))

rng = random.Random(0)
big = mm.schoolbook_4x4()
A = [[rng.randint(-3, 3) for _ in range(4)] for _ in range(4)]
B = [[rng.randint(-3, 3) for _ in range(4)] for _ in range(4)]
print("schoolbook_4x4 exact (random)?", verify_exact(big, A, B))

## 3. Nonequivalence of 4x4 algorithms

The verifier decides under a bounded transformation group. It never uses
numerical tolerance: a rank/dimension mismatch *proves* nonequivalence, a
successful search *proves* equivalence, otherwise it reports INCONCLUSIVE.

In [ ]:
report = mm.check_equivalence(mm.schoolbook_4x4(), mm.strassen_4x4())
print("result  :", report.result.value)
print("evidence:", report.evidence)
print("invariant:", report.invariant)

## 4. Recombination

Smaller factorizations compose into larger ones. `compose_strassen` builds the
rank-49 4x4 Strassen algorithm from the rank-7 2x2 base; provenance survives
serialization and can be inspected / verified.

In [ ]:
composed = mm.compose_strassen(mm.registry.get("strassen_2x2"))
print("dims:", composed.dims, "rank:", composed.rank)
print("recomposition verified?", mm.verify_decomposition(composed, mm.registry.get("strassen_2x2")))
print("operation_count:", mm.operation_count(composed))

## 5. Optional V100 benchmarking

CPU benchmarking works everywhere. CUDA requires the optional `gpu` dependency
group and a real GPU; the runner refuses to report CPU timings as GPU results.
Set `RUN_GPU_TESTS=1` only on a CUDA machine.

In [ ]:
import json

from alphakhulnasoft.matmul.benchmarking.runner import run_benchmark

result = run_benchmark(
    mm.registry.get("strassen_2x2"),
    dims=(2, 2, 2),
    device="cpu",
    batch_size=8,
    warmups=3,
    repetitions=20,
)
print(json.dumps(result.as_dict(), indent=2))

### GPU invocation (run only where CUDA + a V100 are present)
```
uv run python -m alphakhulnasoft.matmul.benchmarking.runner \
  --algorithm strassen_2x2 --dims 2 2 2 --device cuda --verify-device v100
```